# Zang RL on a free Colab GPU: balance / walk policy with MJX + brax PPO

Trains the Zang humanoid's learned controller on a **Colab T4 GPU**: thousands of parallel MuJoCo (MJX) worlds, PPO from brax.
It is the **same task** as the CPU code in `ai_stack/rl` (one shared `zang_rl/core.py`): the same 50-value **IMU-free** observation (hardware v0.0.1 has no IMU), the same 15 servo-target actions in `robot.json` order, the same rewards, domain randomisation, pushes and servo command path (latency, ramp, speed cap, zero error).

**What you need:** `zang_rl_colab_bundle.zip` (built on the PC with `make bundle` in `ai_stack/rl`; about 1 MB), and a GPU runtime: *Runtime > Change runtime type > T4 GPU*.

| step | what | time on a T4 (ESTIMATES, not measured) |
|---|---|---|
| 1 | check the GPU | seconds |
| 2 | upload the zip, unzip, verify every file against `MANIFEST.json` | 1 min |
| 3 | install pinned packages (jax CUDA 12, mujoco, mjx, brax, ...) | 3-6 min |
| 4 | build the MJX env, assert it matches the CPU env, measure throughput | 2-5 min (mostly XLA compile) |
| 5 | train with brax PPO, checkpoints to Drive, resumable | stand 50 M steps: about 0.5-3 h; walk 150 M steps: about 2-8 h |
| 6 | learning curves | seconds |
| 7 | export ONNX + C header, evaluate with CPU MuJoCo `sim2sim.py` | 5-20 min |
| 8 | zip the results and download | seconds |

The time column is a guess from public MJX benchmarks (roughly 5 k to 25 k env steps per second on a T4 for a robot this size, 10 physics substeps per policy step). Step 4 **measures** your real number, and step 5 prints an ETA from the measured speed.

**Free Colab sessions can disconnect.** Checkpoints go to Google Drive every `CHECKPOINT_EVERY` steps. To continue after a disconnect, run all cells again with the same `RUN_NAME`: step 5 resumes from the latest checkpoint. Put the zip on Drive once and use `UPLOAD_MODE = "drive"` in step 2 to skip the upload.

## Step 1. Check the GPU

Fails with a clear message if the runtime has no GPU. `jax.devices()` is probed in a separate process here, because Step 3 replaces jax with the pinned version (the pinned build is checked again there).

In [ ]:
import platform, shutil, subprocess, sys

print("python", sys.version.split()[0], "|", platform.platform())
if sys.version_info < (3, 12):
    raise RuntimeError(f"Python {sys.version.split()[0]}: the pinned jax 0.12 needs Python >= 3.12. "
                       "Use a current Colab runtime image.")

smi = shutil.which("nvidia-smi")
if smi is None:
    raise RuntimeError("NO GPU RUNTIME: nvidia-smi was not found. Open Runtime > Change runtime type, choose "
                       "'T4 GPU' as the hardware accelerator, Save, then Runtime > Run all.")
res = subprocess.run([smi], capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    raise RuntimeError("nvidia-smi failed: " + (res.stderr or res.stdout).strip() +
                       "\nThe runtime has no usable GPU. Choose Runtime > Change runtime type > T4 GPU.")
gpu = subprocess.run([smi, "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu)
if "T4" not in gpu:
    print("note: this notebook is sized for a T4 (16 GB). Another GPU works, but NUM_ENVS in Step 5 may need changing.")

probe = subprocess.run([sys.executable, "-c", "import jax; print(jax.__version__, jax.devices())"],
                       capture_output=True, text=True)
print("jax on the stock image ->", (probe.stdout.strip() or (probe.stderr.strip().splitlines() or ["no output"])[-1]))

## Step 2. Upload the bundle, unzip, verify

Upload `zang_rl_colab_bundle.zip` (file dialog), or copy it from Google Drive (`UPLOAD_MODE = "drive"`). Then **every file is checked against `MANIFEST.json` (sha256)**. The cell prints a pass/fail table and **stops** on any mismatch, missing or unlisted file.

In [ ]:
import importlib.util, os, shutil, zipfile

UPLOAD_MODE = "upload"      # "upload": file dialog  |  "drive": copy from DRIVE_ZIP
DRIVE_ZIP = "/content/drive/MyDrive/zang_rl_colab_bundle.zip"
USE_DRIVE = True            # mount Google Drive: checkpoints and results survive a disconnected session

BUNDLE_ZIP = "/content/zang_rl_colab_bundle.zip"
BUNDLE_DIR = "/content/zang_bundle"
IN_COLAB = importlib.util.find_spec("google.colab") is not None

DRIVE_OK = False
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OK = os.path.isdir("/content/drive/MyDrive")
OUT_ROOT = "/content/drive/MyDrive/zang_rl_runs" if DRIVE_OK else "/content/zang_rl_runs"
print("runs and checkpoints go to:", OUT_ROOT, "" if DRIVE_OK else "(local disk: lost when the session ends)")

if os.path.exists(BUNDLE_ZIP):
    print("reusing", BUNDLE_ZIP)
elif UPLOAD_MODE == "drive":
    if not os.path.exists(DRIVE_ZIP):
        raise FileNotFoundError(f"{DRIVE_ZIP} not found: put the zip there or set UPLOAD_MODE = 'upload'")
    shutil.copy(DRIVE_ZIP, BUNDLE_ZIP)
else:
    if not IN_COLAB:
        raise RuntimeError("not running in Colab: copy the zip to " + BUNDLE_ZIP + " yourself")
    from google.colab import files
    uploaded = files.upload()
    zips = [n for n in uploaded if n.lower().endswith(".zip")]
    if len(zips) != 1:
        raise RuntimeError(f"upload exactly one .zip (got {list(uploaded)})")
    shutil.move(zips[0], BUNDLE_ZIP)

shutil.rmtree(BUNDLE_DIR, ignore_errors=True)
with zipfile.ZipFile(BUNDLE_ZIP) as z:
    root = os.path.realpath(BUNDLE_DIR)
    for n in z.namelist():                                   # refuse paths that escape the target folder
        if not os.path.realpath(os.path.join(root, n)).startswith(root + os.sep):
            raise RuntimeError("unsafe path in zip: " + n)
    z.extractall(BUNDLE_DIR)
print("unzipped to", BUNDLE_DIR, f"({os.path.getsize(BUNDLE_ZIP) / 1e6:.2f} MB zip)")

In [ ]:
import hashlib, json, os

manifest = json.load(open(os.path.join(BUNDLE_DIR, "MANIFEST.json")))
rows, listed = [], set()
for f in manifest["files"]:
    listed.add(f["path"])
    p = os.path.join(BUNDLE_DIR, f["path"])
    if not os.path.isfile(p):
        status = "MISSING"
    else:
        data = open(p, "rb").read()
        ok = hashlib.sha256(data).hexdigest() == f["sha256"] and len(data) == f["size"]
        status = "OK" if ok else "MISMATCH"
    rows.append((status, f["size"], f["path"]))
on_disk = {os.path.relpath(os.path.join(r, n), BUNDLE_DIR).replace(os.sep, "/")
           for r, _, ns in os.walk(BUNDLE_DIR) for n in ns} - {"MANIFEST.json"}
rows += [("UNLISTED", os.path.getsize(os.path.join(BUNDLE_DIR, p)), p) for p in sorted(on_disk - listed)]

print(f"{'status':<9}{'bytes':>9}  file")
for status, size, path in rows:
    print(f"{status:<9}{size:>9}  {path}")
bad = [r for r in rows if r[0] != "OK"]
print(f"\n{len(rows) - len(bad)} / {len(rows)} files OK  ->", "PASS" if not bad else "FAIL")
if bad:
    raise RuntimeError(f"bundle verification FAILED for {len(bad)} file(s) (see the table). "
                       "Re-upload zang_rl_colab_bundle.zip (delete /content/zang_rl_colab_bundle.zip first).")

## Step 3. Install pinned packages

Installs `requirements_colab.txt` from the bundle: `jax[cuda12]` (with its own CUDA libraries), `mujoco`, `mujoco-mjx`, `brax`, `flax`, `onnx` and friends, all pinned to the versions the code was checked against. Colab's own torch is kept (only the export step uses it). Then it prints the versions and **fails if jax cannot see the GPU**.

If it says the runtime must be restarted (a package was already imported in this session): *Runtime > Restart session*, then *Runtime > Run all*. The restart keeps `/content`, so the zip is not uploaded again.

In [ ]:
import importlib.metadata as md
import os, subprocess, sys, time

REQ = os.path.join(BUNDLE_DIR, "requirements_colab.txt")
t0 = time.time()
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", REQ], capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-4000:], r.stderr[-4000:])
    raise RuntimeError("pip install failed (output above)")
print(f"pip install OK in {time.time() - t0:.0f} s")

PROBE = """
import importlib.metadata as md, jax
for p in ("jax", "jaxlib", "jax-cuda12-plugin", "mujoco", "mujoco-mjx", "brax", "flax", "optax", "orbax-checkpoint",
          "gymnasium", "onnx", "onnxruntime", "onnxscript", "numpy", "torch"):
    try:
        print(f"  {p:<20s}{md.version(p)}")
    except md.PackageNotFoundError:
        print(f"  {p:<20s}-")
print("  jax devices:", jax.devices())
assert jax.devices()[0].platform in ("gpu", "cuda"), "jax is running on the CPU"
"""
r = subprocess.run([sys.executable, "-c", PROBE], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print(r.stderr[-3000:])
    raise RuntimeError("jax does not see the GPU with the pinned install (see above). "
                       "Check Step 1 (GPU runtime) and try Runtime > Restart session, then Run all.")

stale = [m for m in ("jax", "jaxlib", "numpy", "flax", "mujoco")
         if m in sys.modules and getattr(sys.modules[m], "__version__", None) != md.version(m)]
if stale:
    raise RuntimeError(f"{stale} were imported before the install and are the old versions in this kernel. "
                       "Runtime > Restart session, then Runtime > Run all (the uploaded zip is kept).")
print("GPU OK, versions as pinned")

## Step 4. The MJX environment, and a check that it equals the CPU one

`zang_rl/mjx_env.py` is the GPU twin of the local environments. Every piece of task maths (domain randomisation, initial state, servo command path, observation, rewards, terminations, commands, pushes) is the **same** `zang_rl/core.py` called with `jax.numpy` instead of `numpy`. Only stepping differs: `mjx.step` over 10 substeps of 2 ms. Collision geometry: a box under each foot against the floor plane, no self collision. The closed parallelogram loops stay as the joint-coupling equalities of the MJCF.

The cell below **asserts** that the MJX env and the CPU Gymnasium env (`ZangStand-v0`, `ZangWalk-v0`) agree on the observation size and block layout, the action size and bounds, the joint names and order (`robot.json`), the home pose, the sensor layout, the reward terms and the full config. It also checks that both start in the same state, then measures the env throughput with random actions.

In [ ]:
import dataclasses, os, sys, time

RL = os.path.join(BUNDLE_DIR, "Zang", "ai_stack", "rl")
sys.path.insert(0, RL)
os.chdir(RL)

import gymnasium as gym
import jax, jax.numpy as jnp
import numpy as np

import zang_rl                                    # registers ZangStand-v0 / ZangWalk-v0 (CPU Gymnasium envs)
from zang_rl import core, robot
from zang_rl import obs as O
from zang_rl.config import EnvCfg
from zang_rl.gym_env import ZangGymEnv
from zang_rl.mjx_env import ZangMjxEnv, wrap_for_brax

assert jax.devices()[0].platform in ("gpu", "cuda"), "jax is not on the GPU: " + str(jax.devices())
print("jax", jax.__version__, jax.devices())

joint_names = [j["name"] for j in robot.load_joint_table()["joints"]]       # ai_stack/config/robot.json, wire order
blocks = list(O.layout(False))                                              # IMU-free layout
assert blocks == ["q", "dq", "last_action", "command", "phase"] and O.obs_dim(False) == 50

for task, gym_id in (("stand", "ZangStand-v0"), ("joystick", "ZangWalk-v0")):
    mx = ZangMjxEnv(task=task)                    # GPU env, default config (what Step 5 trains)
    cpu = gym.make(gym_id).unwrapped              # local CPU Gymnasium env, default config
    assert mx.observation_size["state"] == (50,) == cpu.observation_space.shape == (O.obs_dim(False),)
    assert mx.observation_size["privileged_state"] == (core.critic_dim(),)
    assert mx.action_size == cpu.action_space.shape[0] == len(joint_names) == 15
    assert float(cpu.action_space.low.min()) == -1.0 and float(cpu.action_space.high.max()) == 1.0
    for e in (mx, cpu):
        assert e.zm.joint_names == joint_names, "joint order differs from robot.json"
    assert np.allclose(mx.zm.home_q, cpu.zm.home_q) and np.array_equal(mx.zm.act_ids, cpu.zm.act_ids)
    assert np.array_equal(mx.zm.qpos_adr, cpu.zm.qpos_adr) and mx.zm.sensor_slices == cpu.zm.sensor_slices
    assert dataclasses.asdict(mx.cfg) == dataclasses.asdict(cpu.cfg), "env config differs"
    assert mx.reward_names == list(cpu.cfg.reward_scales)
    assert mx.cfg.gym_action_gain == cpu.cfg.gym_action_gain and mx.nsub == cpu.nsub == 10
    print(f"{task:>9}: obs {mx.observation_size['state'][0]} {blocks}, critic {mx.observation_size['privileged_state'][0]}, "
          f"action {mx.action_size} (x{mx.cfg.gym_action_gain} gain), {len(mx.reward_names)} reward terms: OK")
print("joint order:", joint_names)

# same start state: no randomisation, noise or pushes, fixed pose and heading, on both backends
def quiet_cfg(task):
    c = EnvCfg(task=task, nworld=1, init_noise_q=0.0, init_yaw=0.0)
    c.dr.enabled = c.noise.enabled = c.push.enabled = False
    return c

cpu_q, mx_q = ZangGymEnv(task="stand", cfg=quiet_cfg("stand")), ZangMjxEnv(cfg=quiet_cfg("stand"))
o_cpu, _ = cpu_q.reset(seed=0)
t0 = time.time()
s0 = jax.jit(mx_q.reset)(jax.random.PRNGKey(0))
d_qpos = float(np.abs(cpu_q.data.qpos - np.asarray(s0.pipeline_state.qpos)).max())
d_obs = float(np.abs(o_cpu - np.asarray(s0.obs["state"])).max())
print(f"same start state: max |qpos cpu - mjx| = {d_qpos:.1e}, max |obs cpu - mjx| = {d_obs:.1e}  (first MJX compile {time.time() - t0:.0f} s)")
assert d_qpos < 1e-4 and d_obs < 1e-3, "MJX and CPU envs start differently"
del cpu_q, mx_q, s0
print("PARITY CHECKS PASSED")

In [ ]:
# Throughput with random actions (includes auto-resets, randomisation, pushes). This is a measured speed
# for THIS runtime; PPO updates make real training somewhat slower.
BENCH_ENVS, BENCH_STEPS = 2048, 30

venv = wrap_for_brax(ZangMjxEnv(task="stand"))
reset, step = jax.jit(venv.reset), jax.jit(venv.step)
rand_act = jax.jit(lambda k: jax.random.uniform(k, (BENCH_ENVS, 15), minval=-1.0, maxval=1.0))
keys = jax.random.split(jax.random.PRNGKey(1), BENCH_ENVS)
t0 = time.time()
st = step(reset(keys), jnp.zeros((BENCH_ENVS, 15)))
jax.block_until_ready(st)
print(f"compile + first step: {time.time() - t0:.0f} s")
k = jax.random.PRNGKey(2)
t0 = time.time()
resets = 0.0
for _ in range(BENCH_STEPS):
    k, ka = jax.random.split(k)
    st = step(st, rand_act(ka))
    resets += float(st.done.mean())
jax.block_until_ready(st)
dt = time.time() - t0
BENCH_SPS = BENCH_ENVS * BENCH_STEPS / dt
print(f"MEASURED: {BENCH_SPS:,.0f} env steps/s with {BENCH_ENVS} envs  ({BENCH_SPS * 0.02:,.0f}x real time)")
print("observations finite:", bool(jnp.isfinite(st.obs["state"]).all() & jnp.isfinite(st.obs["privileged_state"]).all()),
      "| mean reward/step:", f"{float(st.reward.mean()):.4f}", "| random-action episodes ended per step:", f"{resets / BENCH_STEPS:.3f}")
ms = jax.devices()[0].memory_stats() or {}
if ms:
    print(f"GPU memory in use: {ms.get('bytes_in_use', 0) / 1e9:.2f} GB of {ms.get('bytes_limit', 0) / 1e9:.1f} GB")
del st, venv, reset, step
jax.clear_caches()

## Step 5a. Training settings

Edit this cell, then run the training cell below it.

- `TASK`: `"stand"` learns to stand and recover from pushes. `"walk"` learns velocity tracking (`ZangWalk`; commands vx -0.1..0.2 m/s, vy +-0.06 m/s, wz +-0.4 rad/s; no hip yaw, so turning is weak).
- `NUM_ENVS`: parallel worlds on the GPU. Lower it (2048, 1024) if you see an out-of-memory error. Must be a multiple of `NUM_MINIBATCHES`.
- `TOTAL_STEPS`: environment steps (1 step = 20 ms of robot time). Suggested: stand 30-50 M, walk 100-200 M.
- `SEED`, `RUN_NAME`: the run folder is `OUT_ROOT/RUN_NAME`. **Use the same `RUN_NAME` to resume** after a disconnect. Changing `TOTAL_STEPS` and re-running continues to the new total.

The PPO numbers mirror the CPU/rsl_rl settings in `ai_stack/rl/zang_rl/registry.py` (fixed learning rate 3e-4, gamma 0.99, GAE 0.95, entropy 0.005, actor 50-128-128-15 with ELU, the size the ESP32-P4 runs). The CPU runs that used these settings did **not** yet beat simply holding the home pose, so treat the sim2sim table in Step 7 as the real test.

In [ ]:
TASK = "stand"                  # "stand" | "walk"
NUM_ENVS = 4096                 # parallel MJX worlds
TOTAL_STEPS = 50_000_000        # env steps (stand 30-50 M, walk 100-200 M)
SEED = 0
RUN_NAME = ""                   # "" -> f"{TASK}_seed{SEED}"; reuse a name to resume
CHECKPOINT_EVERY = 5_000_000    # env steps between checkpoints (and evaluations)

# PPO (brax)
UNROLL_LENGTH = 24              # steps per env per rollout
NUM_MINIBATCHES = 16            # batch_size = NUM_ENVS // NUM_MINIBATCHES trajectories per minibatch
NUM_UPDATES_PER_BATCH = 4       # SGD passes over each rollout
LEARNING_RATE = 3e-4
ENTROPY_COST = 0.005
DISCOUNTING = 0.99
GAE_LAMBDA = 0.95
CLIPPING_EPSILON = 0.2
MAX_GRAD_NORM = 1.0
INIT_NOISE_STD = 0.8            # initial exploration std of the actions
POLICY_HIDDEN = (128, 128)      # actor: 50-128-128-15 = about 25 k MACs, runs on the ESP32-P4
VALUE_HIDDEN = (256, 256, 128)
NUM_EVAL_ENVS = 128             # episodes per evaluation (deterministic policy, same randomisation and pushes)

assert TASK in ("stand", "walk"), "TASK must be 'stand' or 'walk'"
assert NUM_ENVS % NUM_MINIBATCHES == 0, "NUM_ENVS must be a multiple of NUM_MINIBATCHES"
print(f"TASK={TASK}  NUM_ENVS={NUM_ENVS}  TOTAL_STEPS={TOTAL_STEPS:,}  SEED={SEED}  "
      f"-> {NUM_ENVS * UNROLL_LENGTH:,} env steps per PPO iteration")
try:
    print(f"ESTIMATE of training time: {TOTAL_STEPS / (0.5 * BENCH_SPS) / 3600:.1f} h "
          f"(assumes training runs at half the {BENCH_SPS:,.0f} steps/s measured in Step 4; first epoch adds compile time)")
except NameError:
    print("(run Step 4 to get a measured speed and a time estimate)")

## Step 5b. Train (resumable)

brax PPO with an **asymmetric** actor-critic: the actor sees the 50 IMU-free observations, the critic additionally sees privileged signals (true IMU, base velocity, foot contact, the randomised parameters), used in training only.

Every `CHECKPOINT_EVERY` steps it writes a checkpoint to `OUT_ROOT/RUN_NAME` (atomic, newest 3 kept) and runs an evaluation: `eval return`, `fall` (share of episodes that ended in a fall), and episode length. The first epoch includes the XLA compile (several minutes, estimate). The ETA is printed from the second epoch on, using measured speed.

Stopping with the stop button is safe: run the cell again to resume from the last checkpoint. Optimizer state is not restored, so Adam restarts its moments after a resume.

In [ ]:
import functools, importlib.metadata as md, json, math, time
from brax.training.agents.ppo import train as ppo
from zang_rl import brax_train as bt

SIM_TASK = {"stand": "stand", "walk": "joystick"}[TASK]
RUN_NAME = RUN_NAME or f"{TASK}_seed{SEED}"
RUN_DIR = os.path.join(OUT_ROOT, RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)
CURVES = os.path.join(RUN_DIR, "curves.json")

env = ZangMjxEnv(task=SIM_TASK)
cfg = env.cfg
json.dump(dataclasses.asdict(cfg), open(os.path.join(RUN_DIR, "env_cfg.json"), "w"), indent=2)

# ---- resume ---------------------------------------------------------------------------------------------------
steps_done, restore, curves = 0, None, []
latest = bt.latest_checkpoint(RUN_DIR)
if latest:
    ck = bt.load_checkpoint(latest)
    assert ck["meta"].get("task") == TASK, f"{latest} is a '{ck['meta'].get('task')}' run, TASK is '{TASK}': use another RUN_NAME"
    steps_done, restore = ck["steps"], ck["params"]
    if os.path.exists(CURVES):
        curves = [r for r in json.load(open(CURVES)) if r["steps"] <= steps_done]
    print(f"RESUMING {RUN_NAME} from {os.path.basename(latest)}: {steps_done:,} steps done")
else:
    print(f"starting {RUN_NAME} from scratch")

remaining = TOTAL_STEPS - steps_done
train_cfg = dict(task=TASK, sim_task=SIM_TASK, num_envs=NUM_ENVS, total_steps=TOTAL_STEPS, seed=SEED,
                 checkpoint_every=CHECKPOINT_EVERY, unroll_length=UNROLL_LENGTH, num_minibatches=NUM_MINIBATCHES,
                 num_updates_per_batch=NUM_UPDATES_PER_BATCH, learning_rate=LEARNING_RATE, entropy_cost=ENTROPY_COST,
                 discounting=DISCOUNTING, gae_lambda=GAE_LAMBDA, clipping_epsilon=CLIPPING_EPSILON,
                 max_grad_norm=MAX_GRAD_NORM, init_noise_std=INIT_NOISE_STD, policy_hidden=POLICY_HIDDEN,
                 value_hidden=VALUE_HIDDEN, num_eval_envs=NUM_EVAL_ENVS, resumed_at_steps=steps_done,
                 versions={p: md.version(p) for p in ("jax", "mujoco", "mujoco-mjx", "brax", "flax")})
json.dump(train_cfg, open(os.path.join(RUN_DIR, "train_config.json"), "w"), indent=2)

# ---- callbacks ------------------------------------------------------------------------------------------------
t_run = time.time()
n_epochs = 0

def progress(step, m):
    """Called by brax after every epoch with eval + training metrics."""
    global n_epochs
    row = {k: float(v) for k, v in m.items() if np.ndim(v) == 0}
    row["steps"] = steps_done + int(step)
    curves.append(row)
    json.dump(curves, open(CURVES, "w"))
    g = lambda k: row.get(k, float("nan"))
    eta = ""
    if step > 0:
        n_epochs += 1
        if n_epochs >= 2 and g("training/sps") > 0:
            eta = f" | ETA {(TOTAL_STEPS - row['steps']) / g('training/sps') / 60:6.1f} min"
    print(f"{row['steps'] / 1e6:7.1f} M | eval return {g('eval/episode_reward'):8.2f} | fall rate {g('eval/episode_fall'):4.2f} "
          f"| ep len {g('eval/avg_episode_length') * cfg.ctrl_dt:5.1f} s | {g('training/sps'):9,.0f} steps/s{eta} "
          f"| wall {(time.time() - t_run) / 60:6.1f} min")

def on_params(step, make_policy, params):
    """Called by brax with the current (normalizer, policy, value) params at every epoch end."""
    if step > 0:
        path = bt.save_checkpoint(RUN_DIR, steps_done + int(step), params,
                                  meta={"task": TASK, "seed": SEED, "obs_dim": 50, "act_dim": 15})
        print(f"          checkpoint -> {path}")

# ---- train ----------------------------------------------------------------------------------------------------
if remaining <= 0:
    print(f"already at {steps_done:,} >= TOTAL_STEPS {TOTAL_STEPS:,}: nothing to train (raise TOTAL_STEPS to continue)")
else:
    n_evals = max(2, math.ceil(remaining / CHECKPOINT_EVERY) + 1)
    print(f"training {remaining:,} steps in {n_evals - 1} epochs; the first epoch includes the XLA compile ...")
    try:
        ppo.train(
            environment=env,
            num_timesteps=remaining,
            num_envs=NUM_ENVS,
            episode_length=env.episode_length,
            wrap_env_fn=wrap_for_brax,                  # vmap only: the env resets itself (see mjx_env.py)
            network_factory=bt.network_factory(POLICY_HIDDEN, VALUE_HIDDEN, INIT_NOISE_STD),
            unroll_length=UNROLL_LENGTH,
            batch_size=NUM_ENVS // NUM_MINIBATCHES,
            num_minibatches=NUM_MINIBATCHES,
            num_updates_per_batch=NUM_UPDATES_PER_BATCH,
            learning_rate=LEARNING_RATE,
            entropy_cost=ENTROPY_COST,
            discounting=DISCOUNTING,
            gae_lambda=GAE_LAMBDA,
            clipping_epsilon=CLIPPING_EPSILON,
            max_grad_norm=MAX_GRAD_NORM,
            normalize_observations=True,
            normalize_observations_std_eps=1e-4,
            seed=SEED + steps_done // CHECKPOINT_EVERY,
            num_evals=n_evals,
            num_eval_envs=NUM_EVAL_ENVS,
            deterministic_eval=True,
            progress_fn=progress,
            policy_params_fn=on_params,
            restore_params=restore,
            restore_value_fn=True,
        )
    except KeyboardInterrupt:
        print("interrupted: the last checkpoint is safe, run this cell again to resume")

latest = bt.latest_checkpoint(RUN_DIR)
assert latest, "no checkpoint was written (training did not finish a single epoch)"
FINAL = bt.load_checkpoint(latest)
print(f"\nlatest checkpoint: {os.path.basename(latest)}  ({FINAL['steps']:,} steps)  in {RUN_DIR}")

## Step 6. Learning curves

Evaluation return, fall rate and episode length (deterministic policy, same randomisation, noise and pushes as in training), the evaluation return per reward term, the value loss and KL, and the action standard deviation (exploration). The PNG is saved in the run folder.

Reading it: a stand policy is only useful if the fall rate drops towards 0 *and* the sim2sim table (Step 7) beats the home-pose baseline.

In [ ]:
import json, os
import matplotlib.pyplot as plt
import numpy as np

curves = json.load(open(os.path.join(RUN_DIR, "curves.json")))
def series(key):
    pts = [(r["steps"] / 1e6, r[key]) for r in curves if key in r and np.isfinite(r[key])]
    return (np.array([p[0] for p in pts]), np.array([p[1] for p in pts]))

fig, ax = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
x, y = series("eval/episode_reward"); _, sd = series("eval/episode_reward_std")
ax[0, 0].plot(x, y, color="tab:blue"); ax[0, 0].set(title="evaluation return per episode", ylabel="return")
if len(sd) == len(y): ax[0, 0].fill_between(x, y - sd, y + sd, color="tab:blue", alpha=0.2)
x, y = series("eval/episode_fall"); ax[0, 1].plot(x, y, color="tab:red"); ax[0, 1].set(title="fall rate (share of episodes ending in a fall)", ylim=(-0.02, 1.02))
x, y = series("eval/avg_episode_length"); ax[0, 2].plot(x, y * cfg.ctrl_dt, color="tab:green"); ax[0, 2].set(title="episode length (s), max 20 s")
terms = sorted({k for r in curves for k in r if k.startswith("eval/episode_reward/") and not k.endswith("_std")})
for k in terms:
    x, y = series(k); ax[1, 0].plot(x, y, label=k.split("/")[-1])
ax[1, 0].set(title="evaluation return per reward term"); ax[1, 0].legend(fontsize=7, ncol=2)
x, y = series("training/v_loss"); ax[1, 1].plot(x, y, color="tab:purple", label="value loss"); ax[1, 1].set_yscale("log")
a2 = ax[1, 1].twinx(); x, y = series("training/kl_mean"); a2.plot(x, y, color="tab:orange", label="KL"); a2.set_ylabel("KL", color="tab:orange")
ax[1, 1].set(title="value loss and policy KL")
x, y = series("training/policy_dist_mean_std"); ax[1, 2].plot(x, y, color="tab:brown"); ax[1, 2].set(title="mean action std (exploration)")
for a in ax.ravel():
    a.set_xlabel("env steps (millions)"); a.grid(alpha=0.3)
fig.suptitle(f"{RUN_NAME}: {TASK}, {NUM_ENVS} envs, seed {SEED}")
png = os.path.join(RUN_DIR, "learning_curves.png")
fig.savefig(png, dpi=120)
plt.show()
sps = [r["training/sps"] for r in curves if r.get("training/sps", 0) > 0]
if sps:
    print(f"measured training speed: median {np.median(sps):,.0f} env steps/s over {len(sps)} epochs (the first includes compile time)")
print("saved", png)

## Step 7a. Export: ONNX and the C header

Writes the actor (observation normaliser folded in) with `export_policy.py`: `policy.onnx`, `policy_meta.json` and `p4/policy_weights.h` for the ESP32-P4 C MLP. It then compiles the C MLP with gcc and compares it with onnxruntime on 200 random observations.

The brax actor output goes through `clip(-1, 1) * 2` (the env's action convention), exactly as the P4 C code does; target = home + 0.3 rad * action, wire order of `robot.json`. The exported ONNX is what the next cell evaluates, so the test is of the artifact that goes on the robot.

In [ ]:
import subprocess, sys

NPZ = os.path.join(RUN_DIR, "policy_brax.npz")
FINAL = bt.load_checkpoint(bt.latest_checkpoint(RUN_DIR))
bt.export_actor_npz(FINAL["params"], NPZ)
print("actor weights ->", NPZ, f"(checkpoint at {FINAL['steps']:,} steps)")

r = subprocess.run([sys.executable, "export_policy.py", NPZ], cwd=RL, capture_output=True, text=True)
print(r.stdout.strip()[-1500:])
ONNX = os.path.join(RUN_DIR, "policy.onnx")
HEADER = os.path.join(RUN_DIR, "p4", "policy_weights.h")
if r.returncode:
    print(r.stderr.strip()[-2500:])
    ok_files = os.path.exists(ONNX) and os.path.exists(HEADER)
    raise RuntimeError("export_policy.py failed" + (" in the C-vs-ONNX check (ONNX and header were written, "
                       "but do not use them before the mismatch is understood)" if ok_files else " (see above)"))
meta = json.load(open(os.path.join(RUN_DIR, "policy_meta.json")))
print(f"policy: task={meta['task']}, obs {meta['obs_dim']} (IMU: {meta['imu']}), actions {meta['act_dim']}, {meta['mlp_macs']:,} MACs per step")
print("files:", ONNX, HEADER, sep="\n  ")

## Step 7b. Evaluate with CPU MuJoCo (`sim2sim.py`)

`sim2sim.py` is an independent control loop, written like the P4 firmware, run in plain CPU MuJoCo (not MJX) on the exported ONNX. Two physics settings: `train` (2 ms, implicit-fast, feet-only collision: what the policy trained on) and `sim2sim` (1 ms, RK4, full mesh collision: physics it never saw). Sensor noise is on, with a push every 2.5 s at each listed speed.

Columns to read: `fall_rate`, `tilt_mean_deg` / `tilt_max_deg`, `track_rms_deg` (commanded vs measured joint angle), and for walking `vel_err_mps` / `mean_vx_mps` (velocity command tracking). **The home-pose baseline** (zero action, the robot just holds its crouch) is run at the same pushes: a learned policy is only worth deploying if it beats it.

Runtime: roughly 5-20 minutes on Colab's 2 CPU cores (estimate; more for `walk`, which tests two commands). Reduce `EVAL_EPISODES` or `EVAL_PUSHES` to go faster.

In [ ]:
EVAL_EPISODES = 6
EVAL_SECONDS = 10
EVAL_PUSHES = (0.45, 0.6)                       # m/s pelvis kick every 2.5 s (training pushes are 0..0.6)
WALK_COMMANDS = ((0.0, 0.0, 0.0), (0.10, 0.0, 0.0))     # only used when TASK == "walk"

commands = ((0.0, 0.0, 0.0),) if TASK == "stand" else WALK_COMMANDS
report = []
def run_sim2sim(policy, push, command):
    cmd = ["--command", *map(str, command)] if TASK == "walk" else []
    args = [sys.executable, "sim2sim.py", "--policy", policy, "--configs", "train,sim2sim", "--episodes", str(EVAL_EPISODES),
            "--seconds", str(EVAL_SECONDS), "--push", str(push), *cmd]
    out = subprocess.run(args, cwd=RL, capture_output=True, text=True)
    text = out.stdout.strip() if out.returncode == 0 else out.stdout + out.stderr
    report.append(text)
    print(text, "\n")

t0 = time.time()
for command in commands:
    for push in EVAL_PUSHES:
        label = f"command {command}" if TASK == "walk" else "no command"
        print(f"################ LEARNED policy  push {push} m/s  {label}")
        run_sim2sim(ONNX, push, command)
        print(f"################ BASELINE: hold the home pose (zero action)  push {push} m/s  {label}")
        run_sim2sim("home", push, command)
open(os.path.join(RUN_DIR, "sim2sim_report.txt"), "w").write("\n\n".join(report) + "\n")
print(f"done in {(time.time() - t0) / 60:.1f} min, report saved to {os.path.join(RUN_DIR, 'sim2sim_report.txt')}")

## Step 8. Zip the results and download

Packs the policy (`policy_brax.npz`, `policy.onnx`, `policy_meta.json`, `p4/policy_weights.h`), the learning curves (PNG and `curves.json`), the configs (`env_cfg.json`, `train_config.json`), the sim2sim report and the latest checkpoint (to continue training later with the same `RUN_NAME`). The zip is also kept in the run folder on Drive.

Next on the PC: put `policy_weights.h` where `ai_stack/rl/export/P4_POLICY.md` says, and re-check with `sim2sim.py` before anything runs on the robot. The IMU-free policy is expected to be weak at push recovery; that is a hardware limit (see the README), not only a training one.

In [ ]:
import glob, shutil, zipfile

latest_ckpt = bt.latest_checkpoint(RUN_DIR)
names = ["policy_brax.npz", "policy.onnx", "policy_meta.json", os.path.join("p4", "policy_weights.h"),
         "learning_curves.png", "curves.json", "env_cfg.json", "train_config.json", "sim2sim_report.txt",
         os.path.basename(latest_ckpt)]
zip_path = os.path.join(RUN_DIR, f"zang_{RUN_NAME}_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for n in names:
        p = os.path.join(RUN_DIR, n)
        if os.path.exists(p):
            z.write(p, os.path.join(RUN_NAME, n))
            print(f"  + {n} ({os.path.getsize(p) / 1e3:.0f} kB)")
        else:
            print(f"  - {n} missing (skipped)")
print(f"\n{zip_path}  {os.path.getsize(zip_path) / 1e6:.2f} MB")
if IN_COLAB:
    from google.colab import files
    local_copy = os.path.join("/content", os.path.basename(zip_path))
    shutil.copy(zip_path, local_copy)
    files.download(local_copy)
else:
    print("not in Colab: copy the zip from the path above")